# Speech Emotion Recognition — Evaluation Notebook

Loads pre-trained weights and evaluates on RAVDESS, TESS, CREMA-D, SAVEE test split.

**Setup:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `best_ser_v2.keras`, `label_encoder.pkl`, `scaler.pkl` to Google Drive under `MyDrive/SER/`
3. Get your Kaggle API token: kaggle.com → Settings → API → Create New Token
4. Run All

In [ ]:
# ── Cell 1: Install + mount Drive ────────────────────────────────────────────
!pip install librosa kaggle -q

from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/SER/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive mounted. Output dir:', OUTPUT_DIR)

In [ ]:
# ── Cell 2: Download datasets ─────────────────────────────────────────────────
# Paste your Kaggle API token below (kaggle.com → Settings → API → Create New Token)
# It looks like: KGAT_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
import os

KAGGLE_TOKEN = 'PASTE_YOUR_TOKEN_HERE'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

DATA_DIR = '/content/datasets'
os.makedirs(DATA_DIR, exist_ok=True)

datasets = [
    ('uwrfkaggler/ravdess-emotional-speech-audio', 'ravdess'),
    ('ejlok1/toronto-emotional-speech-set-tess',   'tess'),
    ('ejlok1/cremad',                              'cremad'),
    ('ejlok1/surrey-audiovisual-expressed-emotion-savee', 'savee'),
]

for slug, name in datasets:
    dest = f'{DATA_DIR}/{name}'
    if os.path.exists(dest) and len(os.listdir(dest)) > 0:
        print(f'  {name}: already downloaded, skipping')
        continue
    print(f'  Downloading {name}...')
    os.makedirs(dest, exist_ok=True)
    os.system(f'kaggle datasets download -d {slug} -p {dest} --unzip -q')
    print(f'  {name}: done')

RAVDESS_PATH = f'{DATA_DIR}/ravdess'
TESS_PATH    = f'{DATA_DIR}/tess'
CREMAD_PATH  = f'{DATA_DIR}/cremad'
SAVEE_PATH   = f'{DATA_DIR}/savee'

for name, path in [('RAVDESS',RAVDESS_PATH),('TESS',TESS_PATH),('CREMA-D',CREMAD_PATH),('SAVEE',SAVEE_PATH)]:
    print(f'  {name}: {"OK" if os.path.exists(path) else "MISSING"}')

In [ ]:
# ── Cell 3: Imports + constants + class definitions ───────────────────────────
import glob, pickle, warnings
import numpy as np
import librosa
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

SAMPLE_RATE    = 22050
DURATION       = 3.0
N_SAMPLES      = int(SAMPLE_RATE * DURATION)
N_MFCC         = 40
N_FFT          = 2048
HOP_LENGTH     = 512
N_MELS         = 128
TARGET_TIME    = int(np.ceil(N_SAMPLES / HOP_LENGTH)) + 1
REG            = regularizers.l2(1e-4)
VALID_EMOTIONS = {'neutral','calm','happy','sad','angry','fear','disgust','surprise'}

class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, smoothing=0.1, **kw):
        super().__init__(**kw)
        self.gamma = gamma; self.smoothing = smoothing
    def call(self, y_true, y_pred):
        nc = tf.cast(tf.shape(y_true)[-1], tf.float32)
        ys = y_true*(1-self.smoothing) + self.smoothing/nc
        yp = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        pt = tf.reduce_sum(y_true*yp, axis=-1, keepdims=True)
        return tf.reduce_mean(tf.pow(1-pt, self.gamma) * (-ys*tf.math.log(yp)))
    def get_config(self):
        c = super().get_config(); c.update({'gamma':self.gamma,'smoothing':self.smoothing}); return c

class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak, wu_steps, total, min_lr=1e-6):
        super().__init__()
        self.peak=float(peak); self.wu=float(wu_steps)
        self.total=float(total); self.min_lr=float(min_lr)
    def __call__(self, step):
        s   = tf.cast(step, tf.float32)
        wu  = self.peak * s / self.wu
        cos = self.min_lr + (self.peak-self.min_lr)*0.5*(1+tf.cos(np.pi*(s-self.wu)/(self.total-self.wu)))
        return tf.where(s < self.wu, wu, cos)
    def get_config(self):
        return {'peak':self.peak,'wu_steps':self.wu,'total':self.total,'min_lr':self.min_lr}

def spec_augment(mel3, fm=15, tm=25, n=2):
    m = mel3.copy(); h, w = m.shape[:2]
    for _ in range(n):
        f  = np.random.randint(1, fm+1); f0 = np.random.randint(0, max(1,h-f))
        m[f0:f0+f, :, :] = 0
        t  = np.random.randint(1, tm+1); t0 = np.random.randint(0, max(1,w-t))
        m[:, t0:t0+t, :] = 0
    return m

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Cell 4: Feature extraction functions + dataset parsers ────────────────────
def load_audio(fp):
    a, _ = librosa.load(fp, sr=SAMPLE_RATE, duration=DURATION)
    a, _ = librosa.effects.trim(a, top_db=25)
    return np.pad(a, (0, max(0, N_SAMPLES-len(a))))[:N_SAMPLES]

def extract_features(audio):
    feats = []
    mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    for fn in [np.mean, np.std, np.min, np.max]: feats.extend(fn(mfcc, axis=1))
    for order in [1, 2]:
        d = librosa.feature.delta(mfcc, order=order)
        feats.extend(np.mean(d,axis=1)); feats.extend(np.std(d,axis=1))
    chroma = librosa.feature.chroma_stft(y=audio, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats.extend(np.mean(chroma,axis=1)); feats.extend(np.std(chroma,axis=1))
    mel = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mdb = librosa.power_to_db(mel, ref=np.max)
    feats.extend(np.mean(mdb,axis=1)); feats.extend(np.std(mdb,axis=1))
    contrast = librosa.feature.spectral_contrast(y=audio, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats.extend(np.mean(contrast,axis=1)); feats.extend(np.std(contrast,axis=1))
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(audio), sr=SAMPLE_RATE)
    feats.extend(np.mean(tonnetz,axis=1)); feats.extend(np.std(tonnetz,axis=1))
    zcr = librosa.feature.zero_crossing_rate(audio)
    feats.extend([np.mean(zcr), np.std(zcr), np.max(zcr)])
    rms = librosa.feature.rms(y=audio)
    feats.extend([np.mean(rms), np.std(rms), np.max(rms)])
    for fn in [librosa.feature.spectral_centroid, librosa.feature.spectral_bandwidth, librosa.feature.spectral_rolloff]:
        f = fn(y=audio, sr=SAMPLE_RATE)
        feats.extend([np.mean(f), np.std(f), np.max(f)])
    pitches, _ = librosa.piptrack(y=audio, sr=SAMPLE_RATE)
    pv = pitches[pitches > 0]
    feats.extend([np.mean(pv), np.std(pv), np.min(pv), np.max(pv)] if len(pv) > 0 else [0,0,0,0])
    return np.array(feats, dtype=np.float32)

def extract_mel_multiscale(audio):
    channels = []
    for nm, nf, hl in [(N_MELS, N_FFT, HOP_LENGTH),
                        (N_MELS, N_FFT//2, HOP_LENGTH//2),
                        (N_MELS, N_FFT*2,  HOP_LENGTH*2)]:
        mel = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=nm, n_fft=nf, hop_length=hl)
        mdb = librosa.power_to_db(mel, ref=np.max)
        mdb = (mdb - mdb.min()) / (mdb.max() - mdb.min() + 1e-8)
        mdb = np.pad(mdb,((0,0),(0,max(0,TARGET_TIME-mdb.shape[1]))))[:,:TARGET_TIME]
        channels.append(mdb)
    return np.stack(channels, axis=-1).astype(np.float32)

def parse_ravdess(base):
    emap = {'01':'neutral','02':'calm','03':'happy','04':'sad',
            '05':'angry','06':'fear','07':'disgust','08':'surprise'}
    paths, labels = [], []
    for f in glob.glob(os.path.join(base,'**/*.wav'), recursive=True):
        parts = os.path.basename(f).split('-')
        if len(parts) >= 3:
            e = emap.get(parts[2])
            if e in VALID_EMOTIONS: paths.append(f); labels.append(e)
    return paths, labels

def parse_tess(base):
    emap = {'angry':'angry','disgust':'disgust','fear':'fear','happy':'happy',
            'neutral':'neutral','sad':'sad','ps':'surprise'}
    paths, labels = [], []
    for f in glob.glob(os.path.join(base,'**/*.wav'), recursive=True):
        stem = os.path.splitext(os.path.basename(f))[0].lower()
        e = emap.get(stem.split('_')[-1])
        if e: paths.append(f); labels.append(e)
    return paths, labels

def parse_cremad(base):
    emap = {'ANG':'angry','DIS':'disgust','FEA':'fear',
            'HAP':'happy','NEU':'neutral','SAD':'sad'}
    paths, labels = [], []
    for f in glob.glob(os.path.join(base,'**/*.wav'), recursive=True):
        parts = os.path.basename(f).split('_')
        if len(parts) >= 3:
            e = emap.get(parts[2])
            if e: paths.append(f); labels.append(e)
    return paths, labels

def parse_savee(base):
    emap = {'a':'angry','d':'disgust','f':'fear',
            'h':'happy','n':'neutral','sa':'sad','su':'surprise'}
    paths, labels = [], []
    for f in glob.glob(os.path.join(base,'**/*.wav'), recursive=True):
        stem   = os.path.splitext(os.path.basename(f))[0].lower()
        prefix = ''.join(c for c in stem if not c.isdigit())
        e = emap.get(prefix)
        if e: paths.append(f); labels.append(e)
    return paths, labels

print('Parsing datasets...')
all_paths, all_labels = [], []
for parser, path in [(parse_ravdess, RAVDESS_PATH), (parse_tess, TESS_PATH),
                     (parse_cremad,  CREMAD_PATH),  (parse_savee, SAVEE_PATH)]:
    if os.path.exists(path):
        p, l = parser(path)
        all_paths.extend(p); all_labels.extend(l)
        print(f'  {os.path.basename(path)}: {len(p)} files')
print(f'Total: {len(all_paths)} files')
print(f'Distribution: {Counter(all_labels)}')

In [ ]:
# ── Cell 5: Extract features (skips if already saved to Drive) ────────────────
flat_cache = OUTPUT_DIR + 'X_flat_base.npy'
spec_cache = OUTPUT_DIR + 'X_2d_base.npy'
y_cache    = OUTPUT_DIR + 'y_base.npy'

if os.path.exists(flat_cache) and os.path.exists(spec_cache) and os.path.exists(y_cache):
    print('Loading cached features from Drive...')
    X_flat_base = np.load(flat_cache)
    X_2d_base   = np.load(spec_cache)
    y_base      = list(np.load(y_cache))
    print(f'Loaded. Shape: flat={X_flat_base.shape}, spec={X_2d_base.shape}')
else:
    print('Extracting features (~8 min)...')
    X_flat_base, X_2d_base, y_base = [], [], []
    for i, (path, label) in enumerate(zip(all_paths, all_labels)):
        try:
            audio = load_audio(path)
            X_flat_base.append(extract_features(audio))
            X_2d_base.append(extract_mel_multiscale(audio))
            y_base.append(label)
        except: pass
        if (i+1) % 1000 == 0: print(f'  {i+1}/{len(all_paths)}')
    X_flat_base = np.array(X_flat_base, dtype=np.float32)
    X_2d_base   = np.array(X_2d_base,   dtype=np.float32)
    try:
        np.save(flat_cache, X_flat_base)
        np.save(spec_cache, X_2d_base)
        np.save(y_cache, np.array(y_base))
        print('Features saved to Drive for next time.')
    except Exception as e:
        print(f'Save skipped: {e} — continuing anyway')

le    = LabelEncoder()
y_int = le.fit_transform(y_base)
y_cat = to_categorical(y_int)
NC    = len(le.classes_)
print(f'Classes ({NC}): {le.classes_}')

In [ ]:
# ── Cell 6: Split + scale ─────────────────────────────────────────────────────
idx = np.arange(len(y_base))
idx_tr, idx_tmp = train_test_split(idx, test_size=0.15, stratify=y_int, random_state=42)
idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.5, stratify=y_int[idx_tmp], random_state=42)

scaler = StandardScaler().fit(X_flat_base[idx_tr])
X_te_f = scaler.transform(X_flat_base[idx_te])
X_te_2 = X_2d_base[idx_te]
y_te   = y_cat[idx_te]
print(f'Test set: {len(idx_te)} samples')

In [ ]:
# ── Cell 7: Build model + load weights ───────────────────────────────────────
# Uses custom layer subclasses instead of Lambda to avoid Keras 3 serialization issues

class ChannelMean(tf.keras.layers.Layer):
    def call(self, x): return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(tf.keras.layers.Layer):
    def call(self, x): return tf.reduce_max(x, axis=-1, keepdims=True)

class AttnWeightedSum(tf.keras.layers.Layer):
    def call(self, inputs): return tf.reduce_sum(inputs[0] * inputs[1], axis=1)

def se_block(x, r=16):
    c   = x.shape[-1]
    gap = layers.GlobalAveragePooling2D()(x)
    gap = layers.Dense(max(c//r,4), activation='relu')(gap)
    gap = layers.Dense(c, activation='sigmoid')(gap)
    return layers.Multiply()([x, layers.Reshape((1,1,c))(gap)])

def cbam_block(x, r=16):
    x   = se_block(x, r)
    avg = ChannelMean()(x)
    mx  = ChannelMax()(x)
    spa = layers.Conv2D(1, 7, padding='same', activation='sigmoid', use_bias=False)(
              layers.Concatenate(axis=-1)([avg, mx]))
    return layers.Multiply()([x, spa])

def res_block(x, filters, drop):
    sc = layers.BatchNormalization()(layers.Conv2D(filters,1,padding='same',use_bias=False)(x))
    h  = layers.ReLU()(layers.BatchNormalization()(
             layers.Conv2D(filters,3,padding='same',kernel_regularizer=REG,use_bias=False)(x)))
    h  = layers.BatchNormalization()(
             layers.Conv2D(filters,3,padding='same',kernel_regularizer=REG,use_bias=False)(h))
    h  = cbam_block(h)
    h  = layers.ReLU()(layers.Add()([h, sc]))
    return layers.Dropout(drop)(layers.MaxPooling2D(2)(h))

def attn_pool(x):
    c      = x.shape[-1]
    x_flat = layers.Reshape((-1, c))(x)
    attn   = layers.Softmax(axis=1)(layers.Dense(1)(x_flat))
    return AttnWeightedSum()([x_flat, attn])

def build_model(flat_dim, spec_shape, nc):
    si = layers.Input(shape=spec_shape, name='spectrogram')
    x  = layers.ReLU()(layers.BatchNormalization()(
             layers.Conv2D(32,3,padding='same',use_bias=False)(si)))
    x  = res_block(x, 64,  0.20)
    x  = res_block(x, 128, 0.25)
    x  = res_block(x, 256, 0.30)
    x  = res_block(x, 512, 0.35)
    x1 = layers.Dropout(0.5)(layers.ReLU()(layers.BatchNormalization()(
             layers.Dense(256, kernel_regularizer=REG, use_bias=False)(attn_pool(x)))))
    fi = layers.Input(shape=(flat_dim,), name='features')
    f  = layers.Dropout(0.4)(layers.ReLU()(layers.BatchNormalization()(
             layers.Dense(512, kernel_regularizer=REG, use_bias=False)(fi))))
    f2 = layers.Dropout(0.4)(layers.ReLU()(layers.BatchNormalization()(
             layers.Dense(256, kernel_regularizer=REG, use_bias=False)(f))))
    f2 = layers.Add()([f2, layers.Dense(256,use_bias=False)(f)])
    f3 = layers.Dropout(0.3)(layers.ReLU()(layers.BatchNormalization()(
             layers.Dense(128, kernel_regularizer=REG, use_bias=False)(f2))))
    x2 = layers.Add()([f3, layers.Dense(128,use_bias=False)(f2)])
    m    = layers.Concatenate()([x1, x2])
    gate = layers.Dense(384, activation='sigmoid', kernel_regularizer=REG)(m)
    m    = layers.Multiply()([m, gate])
    x = layers.Dropout(0.5)(layers.ReLU()(layers.BatchNormalization()(
            layers.Dense(256, kernel_regularizer=REG, use_bias=False)(m))))
    x = layers.Dropout(0.4)(layers.ReLU()(layers.BatchNormalization()(
            layers.Dense(128, kernel_regularizer=REG, use_bias=False)(x))))
    out = layers.Dense(nc, activation='softmax')(x)
    return models.Model([si, fi], out)

model_path = OUTPUT_DIR + 'best_ser_v2.keras'
if not os.path.exists(model_path):
    raise FileNotFoundError('best_ser_v2.keras not found in MyDrive/SER/. Upload it first.')

best_model = build_model(X_te_f.shape[1], X_te_2.shape[1:], NC)
best_model.load_weights(model_path)
best_model.compile(loss=FocalLoss(gamma=2.0, smoothing=0.1), metrics=['accuracy'])
print('Model loaded successfully')

In [ ]:
# ── Cell 8: TTA Evaluation ────────────────────────────────────────────────────
def tta_predict(mdl, X2, Xf, n=15):
    preds = [mdl.predict({'spectrogram':X2,'features':Xf}, verbose=0)]
    for _ in range(n-1):
        X2a = np.array([spec_augment(X2[j], fm=10, tm=15, n=1)
                        + np.random.randn(*X2[j].shape).astype(np.float32)*0.01
                        for j in range(len(Xf))], dtype=np.float32)
        Xfa = Xf + np.random.randn(*Xf.shape).astype(np.float32)*0.05
        preds.append(mdl.predict({'spectrogram':X2a,'features':Xfa}, verbose=0))
    return np.mean(preds, axis=0)

_, acc_std = best_model.evaluate({'spectrogram':X_te_2,'features':X_te_f}, y_te, verbose=0)
print(f'Standard Test Accuracy : {acc_std*100:.2f}%')

tta_p    = tta_predict(best_model, X_te_2, X_te_f, n=15)
y_te_int = np.argmax(y_te,  axis=1)
tta_int  = np.argmax(tta_p, axis=1)
acc_tta  = np.mean(tta_int == y_te_int)
f1_tta   = f1_score(y_te_int, tta_int, average='macro')
print(f'TTA Test Accuracy      : {acc_tta*100:.2f}%')
print(f'TTA Macro F1           : {f1_tta*100:.2f}%')
print()
print(classification_report(y_te_int, tta_int, target_names=le.classes_))

In [ ]:
# ── Cell 9: Confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(y_te_int, tta_int)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix — TTA Accuracy {acc_tta*100:.1f}%  |  Macro F1 {f1_tta*100:.1f}%',
          fontsize=13, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig(OUTPUT_DIR+'results.png', dpi=150)
plt.show()
print(f'\n=== FINAL RESULTS ===')
print(f'Standard Accuracy : {acc_std*100:.2f}%')
print(f'TTA Accuracy      : {acc_tta*100:.2f}%')
print(f'Macro F1          : {f1_tta*100:.2f}%')
print(f'Results saved to  : {OUTPUT_DIR}results.png')